# 🚀 GPU-Accelerated Vector Database Engine
## Phase 1c — Colab Setup & First GPU Kernel Verification

---

This notebook validates the entire Phase 1 build on real GPU hardware.

### What we do here
| Step | Description |
|------|-------------|
| Cell 1 | Verify GPU device, CUDA version, compute capability |
| Cell 2 | Clone the repo from GitHub |
| Cell 3 | Install system dependencies (cmake, MPI) |
| Cell 4 | Configure and compile with CUDA enabled |
| Cell 5 | Run full test suite — CPU baseline + GPU agreement tests |
| Cell 6 | Throughput benchmark — GPU vs CPU queries per second |
| Cell 7 | Python bindings smoke test |

### Before you start
> ⚠️ **Set your runtime first:** `Runtime > Change runtime type > GPU > A100 (or H100)`  
> Running on CPU will skip all GPU tests and benchmarks.

---

## Cell 1 — GPU Device Info
Confirms we have a real CUDA GPU and checks compute capability.  
- `sm_75` = T4, `sm_80` = A100, `sm_90` = H100  
- All three are supported by our build.

In [1]:
import subprocess

# ── Full nvidia-smi output ───────────────────────────────────────────────────
print('=== GPU Device ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# ── CUDA compiler version ────────────────────────────────────────────────────
print('=== CUDA Compiler (nvcc) ===')
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

# ── Structured device summary ────────────────────────────────────────────────
print('=== Device Summary ===')
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
name, mem, cc = result.stdout.strip().split(',')
print(f'  GPU Name:           {name.strip()}')
print(f'  Total Memory:       {mem.strip()}')
print(f'  Compute Capability: sm_{cc.strip().replace(".", "")}')

# Warn if no GPU
if 'NOTFOUND' in result.stdout or result.returncode != 0:
    print('\n⚠️  WARNING: No GPU detected. Set runtime to GPU and restart.')
else:
    print('\n✅ GPU ready')

=== GPU Device ===
Mon Apr 27 08:54:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   27C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

---
## Cell 2 — Clone Repository
Clones from GitHub. If the repo already exists (e.g. re-running the notebook), pulls latest instead.

In [9]:
import os

REPO_URL = 'https://github.com/uday1o1/GPU-Accelerated-Vector-Database.git'
REPO_DIR = '/content/GPU-Accelerated-Vector-Database'

if os.path.exists(REPO_DIR):
    print('📁 Repo already exists — pulling latest changes...')
    os.chdir(REPO_DIR)
    os.system('git pull')
else:
    print('📥 Cloning repository...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone failed — check the repo URL')
    os.chdir(REPO_DIR)

print(f'\n📂 Working directory: {os.getcwd()}')
print('\n=== Recent commits ===')
os.system('git log --oneline -5')

print('\n=== Repo structure ===')
os.system('find . -not -path "./.git/*" -not -path "./build/*" | sort | head -40')

📁 Repo already exists — pulling latest changes...

📂 Working directory: /content/GPU-Accelerated-Vector-Database

=== Recent commits ===

=== Repo structure ===


0

---
## Cell 3 — Install System Dependencies
Installs CMake (if needed) and OpenMPI headers (used in Phase 4).  
Colab already has CUDA toolkit installed — we just need the build tools.

In [3]:
import subprocess

print('=== Installing system dependencies ===')
print('(This may take ~30 seconds)\n')

# Install cmake and MPI
result = subprocess.run(
    'apt-get install -y cmake ninja-build libopenmpi-dev 2>&1 | tail -5',
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f'apt-get stderr: {result.stderr[-300:]}')

# Verify tools
print('=== Tool versions ===')
for tool in ['cmake --version', 'g++ --version', 'nvcc --version']:
    r = subprocess.run(tool.split(), capture_output=True, text=True)
    first_line = r.stdout.splitlines()[0] if r.stdout else 'NOT FOUND'
    status = '✅' if r.returncode == 0 else '❌'
    print(f'  {status} {first_line}')

=== Installing system dependencies ===
(This may take ~30 seconds)

(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.10.1-1_amd64.deb ...
Unpacking ninja-build (1.10.1-1) ...
Setting up ninja-build (1.10.1-1) ...
Processing triggers for man-db (2.10.2-1) ...

=== Tool versions ===
  ✅ cmake version 3.31.10
  ✅ g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
  ✅ nvcc: NVIDIA (R) Cuda compiler driv

---
## Cell 4 — Configure and Build
CMake will detect CUDA automatically on Colab and enable the GPU code paths.  
You should see `CUDA : TRUE` and `sm_75;sm_80;sm_90` in the configure output.  
Build takes approximately **2-4 minutes** on first run.

In [10]:
import subprocess, os, time

REPO_DIR  = '/content/GPU-Accelerated-Vector-Database'
BUILD_DIR = f'{REPO_DIR}/build'
os.makedirs(BUILD_DIR, exist_ok=True)
os.chdir(BUILD_DIR)

# ── CMake Configure ───────────────────────────────────────────────────────────
print('=== Step 1/2: CMake Configure ===')
result = subprocess.run(
    ['cmake', '..', '-DCMAKE_BUILD_TYPE=Release', '-G', 'Unix Makefiles'],
    capture_output=True, text=True
)
# Print summary lines only
for line in result.stdout.splitlines():
    if any(k in line for k in ['CUDA', 'Version', 'Build type', '===', 'Found', 'compiler']):
        print(f'  {line.strip()}')

if result.returncode != 0:
    print('\n❌ Configure FAILED:')
    print(result.stderr[-3000:])
    raise RuntimeError('CMake configure failed')
else:
    print('\n✅ Configure: OK')

# ── Build ─────────────────────────────────────────────────────────────────────
print('\n=== Step 2/2: Build (grab a coffee ☕) ===')
t0 = time.time()
result = subprocess.run(
    ['cmake', '--build', '.', '-j8'],
    capture_output=True, text=True
)
elapsed = time.time() - t0

# Show last 30 build lines
lines = result.stdout.strip().splitlines()
for line in lines[-30:]:
    print(f'  {line}')

if result.returncode != 0:
    print('\n❌ Build FAILED:')
    print(result.stderr[-3000:])
    raise RuntimeError('Build failed')
else:
    print(f'\n✅ Build: OK in {elapsed:.1f}s')

# Confirm key binaries exist
print('\n=== Build artifacts ===')
for path in [
    f'{BUILD_DIR}/tests/test_vectorstore',
    f'{BUILD_DIR}/tests/test_gpu_kernels',
    f'{BUILD_DIR}/src/libvectordb_core.a',
    f'{BUILD_DIR}/src/libvectordb_cuda.a',
]:
    exists = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {exists}  {os.path.basename(path)}')

=== Step 1/2: CMake Configure ===
  -- CUDA found: /usr/local/cuda/bin/nvcc
  -- === VectorDB Engine ===
  --   Version   : 0.1.0
  --   CUDA      : TRUE
  --   Build type: Release

✅ Configure: OK

=== Step 2/2: Build (grab a coffee ☕) ===
  [ 12%] Built target vectordb_core
  [ 25%] Built target gtest
  [ 43%] Built target vectordb_cuda
  [ 56%] Built target gtest_main
  [ 68%] Built target vectordb_py
  [ 75%] Building CXX object tests/CMakeFiles/test_gpu_kernels.dir/test_gpu_kernels.cpp.o
  [ 87%] Built target test_vectorstore
  [ 93%] Linking CUDA device code CMakeFiles/test_gpu_kernels.dir/cmake_device_link.o
  [100%] Linking CXX executable test_gpu_kernels
  [100%] Built target test_gpu_kernels

✅ Build: OK in 2.5s

=== Build artifacts ===
  ✅  test_vectorstore
  ✅  test_gpu_kernels
  ✅  libvectordb_core.a
  ✅  libvectordb_cuda.a


---
## Cell 5 — Run Full Test Suite
Runs both test binaries:
- `test_vectorstore` — 12 CPU unit tests (same ones that passed on Mac)
- `test_gpu_kernels` — 4 CPU baseline tests + 3 GPU agreement tests (new, first time running on real GPU)

The GPU agreement tests verify that GPU kernel results match CPU results within FP16 tolerance.

In [11]:
import subprocess, os

BUILD_DIR = '/content/GPU-Accelerated-Vector-Database/build'

test_binaries = [
    (f'{BUILD_DIR}/tests/test_vectorstore', 'VectorStore Unit Tests (CPU)'),
    (f'{BUILD_DIR}/tests/test_gpu_kernels', 'GPU Kernel Tests (CPU baseline + GPU agreement)'),
]

all_passed = True
summary = []

for binary, label in test_binaries:
    print(f'\n=== {label} ===')

    if not os.path.exists(binary):
        print(f'❌ Binary not found: {binary}')
        print('   Did Cell 4 build succeed?')
        all_passed = False
        continue

    result = subprocess.run([binary], capture_output=True, text=True)
    print(result.stdout)

    if result.returncode != 0:
        print(f'❌ FAILED — stderr:')
        print(result.stderr)
        all_passed = False
        summary.append(f'❌ {label}')
    else:
        # Parse passed count from gtest output
        for line in result.stdout.splitlines():
            if 'PASSED' in line:
                summary.append(f'✅ {label}: {line.strip()}')

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
for s in summary:
    print(f'  {s}')
print()
if all_passed:
    print('✅ ALL TESTS PASSED — GPU kernels verified against CPU baseline')
else:
    print('❌ SOME TESTS FAILED — check output above')


=== VectorStore Unit Tests (CPU) ===
Running main() from /content/GPU-Accelerated-Vector-Database/build/_deps/googletest-src/googletest/src/gtest_main.cc
[==========] Running 12 tests from 1 test suite.
[----------] Global test environment set-up.
[----------] 12 tests from VectorStoreTest
[ RUN      ] VectorStoreTest.ConstructsCorrectly
[       OK ] VectorStoreTest.ConstructsCorrectly (0 ms)
[ RUN      ] VectorStoreTest.ThrowsOnZeroDim
[       OK ] VectorStoreTest.ThrowsOnZeroDim (0 ms)
[ RUN      ] VectorStoreTest.InsertSingleVector
[       OK ] VectorStoreTest.InsertSingleVector (0 ms)
[ RUN      ] VectorStoreTest.InsertWrongDimThrows
[       OK ] VectorStoreTest.InsertWrongDimThrows (0 ms)
[ RUN      ] VectorStoreTest.InsertBatchAssignsSequentialIds
[       OK ] VectorStoreTest.InsertBatchAssignsSequentialIds (0 ms)
[ RUN      ] VectorStoreTest.L2SearchReturnsClosest
[       OK ] VectorStoreTest.L2SearchReturnsClosest (0 ms)
[ RUN      ] VectorStoreTest.L2SearchRanksByDistance
[  

---
## Cell 6 — GPU vs CPU Throughput Benchmark
Measures **queries per second** for exact KNN search:
- 100,000 vectors, dim=128, K=10, 1000 queries
- GPU is warmed up (10 queries) before timing starts
- Expect **10x-100x speedup** on A100 vs single-core CPU

> Note: This is a flat brute-force search (Phase 1 baseline).  
> HNSW index (Phase 2) will be dramatically faster for large datasets.

In [16]:
import os, subprocess

REPO  = '/content/GPU-Accelerated-Vector-Database'
BUILD = f'{REPO}/build'

benchmark_src = '''
#include <iostream>
#include <chrono>
#include <vector>
#include <random>
#include <iomanip>
#include "core/VectorStore.h"
#include "cuda/GpuVectorStore.h"

using namespace vectordb;
using Clock = std::chrono::high_resolution_clock;

std::vector<float> rand_vec(size_t dim) {
    static std::mt19937 rng(42);
    std::uniform_real_distribution<float> dist(-1.0f, 1.0f);
    std::vector<float> v(dim);
    for (auto& x : v) x = dist(rng);
    return v;
}

int main() {
    const size_t DIM       = 128;
    const size_t N         = 100000;
    const size_t N_QUERIES = 1000;
    const size_t K         = 10;

    std::cout << std::string(60, \'=\') << std::endl;
    std::cout << "  VectorDB Phase 1 Throughput Benchmark" << std::endl;
    std::cout << std::string(60, \'=\') << std::endl;
    std::cout << "  Vectors  : " << N << std::endl;
    std::cout << "  Dim      : " << DIM << std::endl;
    std::cout << "  Queries  : " << N_QUERIES << std::endl;
    std::cout << "  K        : " << K << std::endl;
    std::cout << std::string(60, \'=\') << std::endl << std::endl;

    VectorStoreConfig cfg(DIM, MetricType::L2, N + 1000);
    VectorStoreConfig gpu_cfg(DIM, MetricType::L2, N + 1000);
    gpu_cfg.gpu_enabled = true;

    VectorStore          cpu_store(cfg);
    cuda::GpuVectorStore gpu_store(gpu_cfg);

    std::vector<std::vector<float>> vecs(N);
    for (auto& v : vecs) v = rand_vec(DIM);

    std::cout << "Loading " << N << " vectors..." << std::endl;
    cpu_store.insert_batch(vecs);
    gpu_store.insert_batch(vecs);
    std::cout << "Done." << std::endl << std::endl;

    std::vector<std::vector<float>> queries(N_QUERIES);
    for (auto& q : queries) q = rand_vec(DIM);

    auto t0 = Clock::now();
    for (const auto& q : queries) cpu_store.search(q, K);
    auto t1 = Clock::now();
    double cpu_ms  = std::chrono::duration<double, std::milli>(t1 - t0).count();
    double cpu_qps = N_QUERIES / (cpu_ms / 1000.0);

    for (int i = 0; i < 10; ++i) gpu_store.search(queries[0], K);
    auto t2 = Clock::now();
    for (const auto& q : queries) gpu_store.search(q, K);
    auto t3 = Clock::now();
    double gpu_ms  = std::chrono::duration<double, std::milli>(t3 - t2).count();
    double gpu_qps = N_QUERIES / (gpu_ms / 1000.0);

    std::cout << std::fixed << std::setprecision(1);
    std::cout << "  CPU:  " << (int)cpu_qps << " QPS  (" << cpu_ms << " ms)" << std::endl;
    std::cout << "  GPU:  " << (int)gpu_qps << " QPS  (" << gpu_ms << " ms)" << std::endl;
    std::cout << "  Speedup: " << (gpu_qps / cpu_qps) << "x" << std::endl;
    std::cout << std::string(60, \'=\') << std::endl;
    return 0;
}
'''

os.makedirs(f'{REPO}/benchmarks', exist_ok=True)
src_path = f'{REPO}/benchmarks/throughput_benchmark.cpp'
with open(src_path, 'w') as f:
    f.write(benchmark_src)

# Find CUDA lib path dynamically
cuda_lib = subprocess.run(
    'dirname $(find /usr/local/cuda -name "libcudart.so" 2>/dev/null | head -1)',
    shell=True, capture_output=True, text=True
).stdout.strip()

out_path = f'{BUILD}/throughput_benchmark'
compile_cmd = (
    f'nvcc -O3 -std=c++17 -DCUDA_ENABLED '
    f'-I{REPO}/include '
    f'-I/usr/local/cuda/include '
    f'{src_path} '
    f'{BUILD}/src/libvectordb_core.a '
    f'{BUILD}/src/libvectordb_cuda.a '
    f'-o {out_path} '
    f'-L{cuda_lib} -lcudart'
)

print('Compiling benchmark...')
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(f'❌ Compile failed:\n{result.stderr[-2000:]}')
else:
    print('✅ Compiled\n')
    result = subprocess.run([out_path], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(f'STDERR:\n{result.stderr}')
    if result.returncode != 0:
        print(f'❌ Benchmark crashed with return code {result.returncode}')

Compiling benchmark...
✅ Compiled

  VectorDB Phase 1 Throughput Benchmark
  Vectors  : 100000
  Dim      : 128
  Queries  : 1000
  K        : 10

Loading 100000 vectors...
Done.

  CPU:  57 QPS  (17349.1 ms)
  GPU:  1527 QPS  (654.8 ms)
  Speedup: 26.5x



---
## Cell 7 — Python Bindings Smoke Test
Verifies the `vectordb_py` pybind11 module loads and works correctly from Python.  
This is the interface that will be used in all future Colab demo notebooks.

In [18]:
import sys, random

# Add build output to Python path so the .so module is importable
BUILD_SRC = '/content/GPU-Accelerated-Vector-Database/build/src'
if BUILD_SRC not in sys.path:
    sys.path.insert(0, BUILD_SRC)

print('=== Python Bindings Smoke Test ===')
print(f'Looking for vectordb_py in: {BUILD_SRC}\n')

try:
    import vectordb_py as vdb
    print(f'✅ Module loaded: {vdb.__doc__}\n')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('   Did Cell 4 build succeed and produce vectordb_py*.so?')
    raise

# ── Create store ──────────────────────────────────────────────────────────────
DIM = 128
cfg   = vdb.VectorStoreConfig(DIM, vdb.MetricType.L2, 10000)
store = vdb.VectorStore(cfg)
print(f'Store created: {store.info()}')

# ── Insert vectors ────────────────────────────────────────────────────────────
random.seed(42)
vecs = [[random.gauss(0, 1) for _ in range(DIM)] for _ in range(1000)]
ids  = store.insert_batch(vecs)
print(f'Inserted {len(ids)} vectors  |  first_id={ids[0]}  last_id={ids[-1]}')
print(f'Store after insert: {store.info()}')

# ── Search ────────────────────────────────────────────────────────────────────
query   = [random.gauss(0, 1) for _ in range(DIM)]
results = store.search(query, 5)

print(f'\nTop-5 nearest neighbours (L2 distance):')
print(f'{"Rank":>5}  {"Vector ID":>10}  {"L2 Distance":>12}')
print('-' * 32)
for rank, (vid, dist) in enumerate(results, 1):
    print(f'{rank:>5}  {vid:>10}  {dist:>12.4f}')

# ── Verify distance is monotonically increasing ───────────────────────────────
dists = [r[1] for r in results]
assert dists == sorted(dists), 'Results not sorted by distance!'
print('\n✅ Results correctly sorted by distance')
print('✅ Python bindings: OK')

=== Python Bindings Smoke Test ===
Looking for vectordb_py in: /content/GPU-Accelerated-Vector-Database/build/src

✅ Module loaded: GPU-Accelerated Vector Database Engine — Python bindings

Store created: VectorStore { dim=128 size=0 capacity=10000 metric=L2 gpu=no }
Inserted 1000 vectors  |  first_id=0  last_id=999
Store after insert: VectorStore { dim=128 size=1000 capacity=10000 metric=L2 gpu=no }

Top-5 nearest neighbours (L2 distance):
 Rank   Vector ID   L2 Distance
--------------------------------
    1         763      194.2563
    2         414      201.6838
    3         330      202.7377
    4         937      202.9449
    5         224      204.0058

✅ Results correctly sorted by distance
✅ Python bindings: OK


---
## Phase 1c Results

Fill this in after running all cells above:

| Item | Result |
|------|--------|
| GPU device | |
| GPU memory | |
| Compute capability | |
| CUDA version | |
| Build time | |
| test_vectorstore | / 12 passing |
| test_gpu_kernels (CPU baseline) | / 4 passing |
| test_gpu_kernels (GPU agreement) | / 3 passing |
| CPU QPS | |
| GPU QPS | |
| Speedup | x |
| Python bindings | OK / FAIL |

---
### Next: Phase 2a — HNSW Index Structure
Once all cells above pass, go back to Claude and paste your results table.  
Phase 2a builds the HNSW graph index in C++, designed for GPU memory layout.